# 🇻🇳 VietNerm — Train & Publish Pipeline (Local)

Notebook này thực hiện toàn bộ pipeline trên máy local:

```
Generate data → Train (GPU) → Evaluate → Push to 🤗 HuggingFace Hub
```

## ⚙️ Cấu hình

In [ ]:
# ============================================================
# CẤU HÌNH — CHỈNH SỬA Ở ĐÂY
# ============================================================

# Nếu có file .env, các giá trị này sẽ được ghi đè tự động
HF_USERNAME = "ngocthanhdoan"  # @param {type:"string"}

# Số lượng samples để generate
DATASET_SIZE = 2000  # @param {type:"integer"}

# Số epochs train
TRAIN_EPOCHS = 5  # @param {type:"integer"}

# Batch size
BATCH_SIZE = 16  # @param {type:"integer"}

# Doc types cần train (để trống = tất cả)
DOC_TYPES = []  # @param {type:"raw"}

# Push private repos lên HuggingFace?
HF_PRIVATE = False  # @param {type:"boolean"}

# F1 tối thiểu để cho phép publish
MIN_F1 = 0.2  # @param {type:"number"}

## 1. Setup môi trường

In [ ]:
import os
import subprocess
import sys
import time
import json
import re
from pathlib import Path
from datetime import datetime

gpu_available = False
try:
    result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                           capture_output=True, text=True)
    if result.returncode == 0:
        gpu_info = result.stdout.strip()
        gpu_available = True
        print(f"✅ GPU: {gpu_info}")
    else:
        print("⚠️  Không tìm thấy GPU — train sẽ chạy trên CPU (rất chậm)")
except FileNotFoundError:
    print("⚠️  Không tìm thấy GPU — train sẽ chạy trên CPU (rất chậm)")

print(f"Python: {sys.version}")
print(f"Thời gian: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
print("📦 Cài đặt dependencies...")
%pip install -q -r requirements.txt accelerate python-dotenv
print("✅ Xong")

In [ ]:
# Load cấu hình từ file .env (nếu có)
env_path = Path(".env")
hf_token = None
if env_path.exists():
    print("📄 Đang đọc cấu hình từ .env...")
    with open(env_path, "r") as f:
        for line in f:
            if line.startswith("HF_TOKEN="):
                hf_token = line.strip().split("=")[1]
                os.environ["HF_TOKEN"] = hf_token
            if line.startswith("HF_USERNAME="):
                HF_USERNAME = line.strip().split("=")[1]
    if hf_token: print("✅ Đã tìm thấy HF_TOKEN trong .env")
    else: print("⚠️ Không tìm thấy HF_TOKEN trong .env")
else:
    print("ℹ️ Không thấy file .env, sử dụng cấu hình mặc định hoặc nhập tay.")

In [ ]:
# Đăng nhập HuggingFace
from huggingface_hub import login, HfApi
from getpass import getpass

if not hf_token:
    print("Nhập HuggingFace token (https://huggingface.co/settings/tokens):")
    try:
        hf_token = getpass("HF_TOKEN: ")
    except Exception:
        hf_token = input("HF_TOKEN: ")

if hf_token:
    try:
        login(token=hf_token)
        api = HfApi(token=hf_token)
        print(f"✅ Đã đăng nhập: {api.whoami()['name']}")
    except Exception as e:
        print(f"❌ Thất bại: {e}")
        hf_token = None
else:
    print("⚠️ Không có token, sẽ bỏ qua publish.")

## 2. Discover doc types

In [ ]:
import yaml
import codecs
with codecs.open("registry/documents.yaml", encoding="utf-8") as f:
    registry = yaml.safe_load(f)
all_doc_types = list(registry["documents"].keys())
doc_types = DOC_TYPES if DOC_TYPES else all_doc_types
print(f"📋 Sẽ train {len(doc_types)} doc type(s)")

## 3. Run Pipeline

In [ ]:
results = {}
total_start = time.time()

for i, doc_type in enumerate(doc_types, 1):
    doc_name = registry["documents"][doc_type]["name"]
    print(f"\n{'='*60}\n[{i}/{len(doc_types)}] {doc_type}\n{'='*60}")
    doc_start = time.time()
    doc_result = {"doc_type": doc_type, "name": doc_name, "status": "pending"}

    try:
        # 1. Generate
        print(f"📊 Generating {DATASET_SIZE} samples...")
        subprocess.run([sys.executable, "synthetic/generate_dataset.py", "--doc", doc_type, "--size", str(DATASET_SIZE)], check=True)

        # 2. Train
        print(f"🏋️ Training...")
        subprocess.run([sys.executable, "training/train.py", "--doc", doc_type, "--epochs", str(TRAIN_EPOCHS), "--batch_size", str(BATCH_SIZE)], check=True)

        # 3. Eval
        model_dir = f"models/phobert/{doc_type}"
        best_f1 = 0.0
        state_path = Path(model_dir) / "trainer_state.json"
        if state_path.exists():
            with open(state_path) as f: state = json.load(f)
            for entry in state.get("log_history", []):
                f1 = entry.get("eval_f1", 0.0)
                if f1 > best_f1: best_f1 = f1
        doc_result["f1"] = best_f1
        print(f"🔍 Best F1: {best_f1:.4f}")

        # 4. Push
        if best_f1 >= MIN_F1 and hf_token:
            print(f"🚀 Publishing to Hub...")
            m_repo = f"{HF_USERNAME}/phobert-{doc_type}-ner"
            d_repo = f"{HF_USERNAME}/vietnerm-{doc_type}-dataset"
            p_flag = ["--private"] if HF_PRIVATE else []
            # Truyền token qua sys.executable con
            subprocess.run([sys.executable, "huggingface/push_model.py", "--doc", doc_type, "--repo", m_repo, "--token", hf_token] + p_flag, check=True)
            subprocess.run([sys.executable, "huggingface/push_dataset.py", "--doc", doc_type, "--repo", d_repo, "--token", hf_token] + p_flag, check=True)
            doc_result["status"] = "published"
            doc_result["model_url"] = f"https://huggingface.co/{m_repo}"
        else:
            doc_result["status"] = "evaluated"

    except Exception as e:
        print(f"❌ Error: {e}")
        doc_result["status"] = "error"

    doc_result["time_minutes"] = round((time.time() - doc_start) / 60, 1)
    results[doc_type] = doc_result


## 4. Tổng kết

In [ ]:
print("\n" + "=" * 60 + "\n📊 KẾT QUẢ\n" + "=" * 60)
for dt, r in results.items():
    f1 = f"{r.get('f1', 0):.4f}" if r.get('f1') else "—"
    url = r.get('model_url', '')
    print(f"{r['status']:<12} {dt:<20} F1: {f1}  {url}")